# EXXA — Protoplanetary-Disk Denoising: Week 1 → Week 4 (Master Kaggle Notebook)

**Krishan Yadav · ML4Sci EXXA.** One notebook covering the whole journey, run on Kaggle GPU.

| Week | Content | Section |
|---|---|---|
| 1–2 | Load & visualise `clean.npy`/`dirty.npy`; classical baselines (Gaussian / Median / Wiener); PSNR/SSIM/MSE table | §1 |
| 2–3 | Denoising **Autoencoder** (hybrid MSE+SSIM loss) | §2 |
| 3 | **VAE** (MSE+SSIM+KL) | §3 |
| 3 | Supervised **U-Net** (hybrid loss) + noisy→AE→U-Net visuals | §4 |
| 4 | Conditional **DDPM** diffusion (scaled, **multi-GPU**) + DDIM denoising | §5 |
| — | Unified comparison table across every method | §6 |

All models are evaluated on the **same** held-out 64×64 validation patches, so the final table is
apples-to-apples. Originally planned for a local RTX 2050; migrated to Kaggle for speed (uses both
GPUs of a **T4 ×2** for diffusion).

### Before you run
1. **Accelerator** = *GPU T4 x2* (Settings).  2. **Internet** = *On*.
3. Upload `dirty.npy` + `clean.npy` as a Kaggle Dataset → *Add Input*. The notebook auto-finds it.


## §0. Setup — clone fork, install deps, link data


In [ ]:
import os, sys, subprocess, glob

if os.path.exists('/kaggle'):
    REPO_URL = 'https://github.com/KrishanYadav333/EXXA.git'
    BRANCH   = 'week-4'
    REPO     = '/kaggle/working/EXXA'
    PKG      = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',REPO_URL,REPO], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','pytorch-msssim','torchinfo'], check=True)
    os.makedirs(os.path.join(PKG,'data'), exist_ok=True)
    hits = glob.glob('/kaggle/input/**/dirty.npy', recursive=True)
    if hits:
        sd = os.path.dirname(hits[0])
        for fn in ('dirty.npy','clean.npy'):
            dst = os.path.join(PKG,'data',fn)
            if not os.path.exists(dst):
                try: os.symlink(os.path.join(sd,fn), dst)
                except OSError:
                    import shutil; shutil.copy(os.path.join(sd,fn), dst)
    else:
        print('WARNING: dirty.npy not under /kaggle/input — use *Add Input* to attach your data.')
    os.chdir(os.path.join(PKG,'notebooks'))
    if PKG not in sys.path: sys.path.insert(0, PKG)
print('cwd :', os.getcwd())

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from pytorch_msssim import ssim as ssim_fn

from src.baselines import gaussian_denoise, median_denoise, wiener_denoise
from src.models import DenoisingAutoencoder, DenoisingVAE, DenoisingUNet
from src.models.diffusion_unet import default_diffusion_config
from src.utils.losses import HybridLoss, VAELoss
from src.training.diffusion import DenoisingDiffusion

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU  = torch.cuda.device_count()
print('device :', device, '| GPUs:', N_GPU,
      '|', torch.cuda.get_device_name(0) if N_GPU else '')

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if N_GPU: torch.cuda.manual_seed_all(SEED)

### Tunable run config
Defaults are sized for a reasonable Kaggle "Run All". Bump epochs for stronger results
(diffusion dominates the runtime).


In [ ]:
EPOCHS_AE   = 30
EPOCHS_VAE  = 30
EPOCHS_UNET = 30
EPOCHS_DIFF = 100          # diffusion needs many steps; raise for better samples

PATCH_SIZE       = 64
BATCH_SIZE       = 16      # supervised models
GRAD_ACCUM       = 4
N_BASELINE_IMGS  = 50      # full 600x600 images for the classical baseline table
N_EVAL_PATCHES   = 256     # shared held-out patches for the unified comparison
DDIM_STEPS       = 25

In [ ]:
dirty_all = np.load('../data/dirty.npy').astype(np.float32)
clean_all = np.load('../data/clean.npy').astype(np.float32)
print('dirty', dirty_all.shape, '| clean', clean_all.shape,
      '| range [%.3f, %.3f]' % (dirty_all.min(), dirty_all.max()))

idx = np.arange(len(dirty_all))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=SEED, shuffle=True)
print('train', len(train_idx), '| val', len(val_idx))

## §1. Week 1–2 — Data visualisation & classical baselines


In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(15, 6))
rng = np.random.default_rng(0)
for j, i in enumerate(rng.choice(val_idx, 5, replace=False)):
    ax[0, j].imshow(dirty_all[i], cmap='inferno'); ax[0, j].axis('off')
    ax[1, j].imshow(clean_all[i], cmap='inferno'); ax[1, j].axis('off')
ax[0, 0].set_ylabel('dirty', fontsize=12); ax[1, 0].set_ylabel('clean', fontsize=12)
ax[0, 0].set_title('dirty (noisy)', loc='left'); ax[1, 0].set_title('clean (GT)', loc='left')
plt.tight_layout(); plt.show()

In [ ]:
# Classical baselines on full 600x600 images (skimage metrics, data_range=1.0)
from skimage.metrics import peak_signal_noise_ratio as _psnr
from skimage.metrics import structural_similarity as _ssim

def _m(c, d):
    return (_psnr(c, d, data_range=1.0),
            _ssim(c, d, data_range=1.0),
            float(np.mean((c - d) ** 2)))

methods = {'Noisy': lambda d: d,
           'Gaussian s2': lambda d: gaussian_denoise(d, 2.0),
           'Median 3x3': lambda d: median_denoise(d, 3),
           'Wiener': lambda d: np.nan_to_num(wiener_denoise(d))}
acc = {k: [[], [], []] for k in methods}
for i in val_idx[:N_BASELINE_IMGS]:
    c, d = clean_all[i], dirty_all[i]
    for k, fn in methods.items():
        p, s, m = _m(c, fn(d))
        acc[k][0].append(p); acc[k][1].append(s); acc[k][2].append(m)

print(f"{'Method':<14}{'PSNR':>9}{'SSIM':>9}{'MSE':>11}")
print('-' * 43)
for k, (P, S, M) in acc.items():
    print(f"{k:<14}{np.mean(P):>9.3f}{np.mean(S):>9.4f}{np.mean(M):>11.6f}")

## §1.5 Shared patch pipeline, training loop & evaluator

`PatchDataset` extracts one random 64×64 patch per image with per-patch min-max
normalisation to `[0, 1]` (matching the Week-3 notebooks). `train_supervised` handles the
AE / VAE / U-Net cases; `predict_patches` + `metrics` give the unified evaluation used for
every method below.


In [ ]:
class PatchDataset(Dataset):
    def __init__(self, dirty, clean, ids, ps=PATCH_SIZE):
        self.dirty, self.clean, self.ids, self.ps = dirty, clean, ids, ps
        self._h, self._w = dirty.shape[1], dirty.shape[2]
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        k = self.ids[i]
        r = np.random.randint(0, self._h - self.ps + 1)
        c = np.random.randint(0, self._w - self.ps + 1)
        dp = self.dirty[k, r:r+self.ps, c:c+self.ps]
        cp = self.clean[k, r:r+self.ps, c:c+self.ps]
        lo, hi = dp.min(), dp.max()
        if hi > lo:
            dp = (dp - lo) / (hi - lo)
            cp = np.clip((cp - lo) / (hi - lo), 0.0, 1.0)
        return torch.from_numpy(dp[np.newaxis]).float(), torch.from_numpy(cp[np.newaxis]).float()

train_loader = DataLoader(PatchDataset(dirty_all, clean_all, train_idx),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(PatchDataset(dirty_all, clean_all, val_idx),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Fixed held-out eval patches (same tensors reused for every method)
np.random.seed(123)
eval_ds = PatchDataset(dirty_all, clean_all,
                       np.resize(val_idx, N_EVAL_PATCHES))   # cycle val ids up to N
dirty_eval = torch.stack([eval_ds[i][0] for i in range(N_EVAL_PATCHES)])
clean_eval = torch.stack([eval_ds[i][1] for i in range(N_EVAL_PATCHES)])
print('eval set:', tuple(dirty_eval.shape))

RESULTS = {}   # method -> (PSNR, SSIM, MSE)

def metrics(pred, clean):
    pred = pred.clamp(0, 1); clean = clean.clamp(0, 1)
    mse  = torch.mean((pred - clean) ** 2, dim=(1, 2, 3))
    psnr = 10 * torch.log10(1.0 / torch.clamp(mse, min=1e-10))
    s    = ssim_fn(pred, clean, data_range=1.0, size_average=False)
    return float(psnr.mean()), float(s.mean()), float(mse.mean())

@torch.no_grad()
def predict_patches(model, kind, bs=32):
    model.eval(); outs = []
    for i in range(0, len(dirty_eval), bs):
        d = dirty_eval[i:i+bs].to(device)
        if kind == 'ae':
            p = model(d)
        elif kind == 'vae':
            p = model(d)[0]
        elif kind == 'unet':
            t = torch.zeros(d.size(0), dtype=torch.long, device=device)
            p = torch.sigmoid(model(d, t))
        outs.append(p.cpu())
    return torch.cat(outs)

def train_supervised(model, kind, loss_fn, epochs, lr=1e-3, label=''):
    model.to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)
    tr_hist, va_hist, best = [], [], float('inf')
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad(set_to_none=True); run = 0.0
        for step, (d, c) in enumerate(train_loader, 1):
            d, c = d.to(device), c.to(device)
            if kind == 'ae':
                pred = model(d); total = loss_fn(pred, c)[0]
            elif kind == 'unet':
                t = torch.zeros(d.size(0), dtype=torch.long, device=device)
                pred = torch.sigmoid(model(d, t)); total = loss_fn(pred, c)[0]
            elif kind == 'vae':
                out, mu, lv = model(d); total = loss_fn(out, c, mu, lv)[0]
            (total / GRAD_ACCUM).backward()
            run += total.item() * d.size(0)
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                opt.step(); opt.zero_grad(set_to_none=True)
        tr = run / len(train_loader.dataset)
        # validation loss
        model.eval(); vrun = 0.0
        with torch.no_grad():
            for d, c in val_loader:
                d, c = d.to(device), c.to(device)
                if kind == 'ae':
                    total = loss_fn(model(d), c)[0]
                elif kind == 'unet':
                    t = torch.zeros(d.size(0), dtype=torch.long, device=device)
                    total = loss_fn(torch.sigmoid(model(d, t)), c)[0]
                elif kind == 'vae':
                    out, mu, lv = model(d); total = loss_fn(out, c, mu, lv)[0]
                vrun += total.item() * d.size(0)
        va = vrun / len(val_loader.dataset)
        sched.step(va); tr_hist.append(tr); va_hist.append(va)
        best = min(best, va)
        if ep % 5 == 0 or ep == 1 or ep == epochs:
            print(f'  [{label} {ep:3d}/{epochs}] train {tr:.4f} | val {va:.4f}')
    return tr_hist, va_hist

def plot_curves(tr, va, title):
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(tr)+1), tr, label='train', marker='o', ms=3)
    plt.plot(range(1, len(va)+1), va, label='val', marker='s', ms=3)
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Unified-protocol baselines (same eval patches as the ML models)
de = dirty_eval.numpy()
for name, fn in [('Noisy', lambda x: x),
                 ('Gaussian s2', lambda x: gaussian_denoise(x, 2.0)),
                 ('Median 3x3', lambda x: median_denoise(x, 3)),
                 ('Wiener', lambda x: np.nan_to_num(wiener_denoise(x)))]:
    pred = torch.from_numpy(np.stack([fn(de[k, 0])[None] for k in range(len(de))])).float()
    RESULTS[name] = metrics(pred, clean_eval)
    print(f'{name:<14} PSNR {RESULTS[name][0]:6.3f}  SSIM {RESULTS[name][1]:.4f}  MSE {RESULTS[name][2]:.6f}')

## §2. Week 2–3 — Denoising Autoencoder (hybrid MSE+SSIM loss)


In [ ]:
ae = DenoisingAutoencoder().to(device)
print('AE params:', sum(p.numel() for p in ae.parameters()))
tr, va = train_supervised(ae, 'ae', HybridLoss(0.8, 0.2), EPOCHS_AE, label='AE')
plot_curves(tr, va, 'Autoencoder (hybrid loss)')
RESULTS['Autoencoder'] = metrics(predict_patches(ae, 'ae'), clean_eval)
print('AE eval:', RESULTS['Autoencoder'])

## §3. Week 3 — Variational Autoencoder (MSE+SSIM+KL)


In [ ]:
vae = DenoisingVAE(latent_dim=128).to(device)
print('VAE params:', sum(p.numel() for p in vae.parameters()))
tr, va = train_supervised(vae, 'vae', VAELoss(0.8, 0.2, 1e-3), EPOCHS_VAE, label='VAE')
plot_curves(tr, va, 'VAE (MSE+SSIM+KL)')
RESULTS['VAE'] = metrics(predict_patches(vae, 'vae'), clean_eval)
print('VAE eval:', RESULTS['VAE'])

## §4. Week 3 — Supervised U-Net (hybrid loss)


In [ ]:
unet = DenoisingUNet(str(device))
print('U-Net params:', sum(p.numel() for p in unet.parameters()))
tr, va = train_supervised(unet, 'unet', HybridLoss(0.8, 0.2), EPOCHS_UNET, label='UNet')
plot_curves(tr, va, 'Supervised U-Net (hybrid loss)')
RESULTS['U-Net'] = metrics(predict_patches(unet, 'unet'), clean_eval)
print('U-Net eval:', RESULTS['U-Net'])

In [ ]:
# Visual: clean | noisy | AE | VAE | U-Net  (3 random eval patches)
sel = np.random.default_rng(7).choice(len(dirty_eval), 3, replace=False)
pae  = predict_patches(ae,  'ae')
pvae = predict_patches(vae, 'vae')
pun  = predict_patches(unet,'unet')
cols = ['clean', 'noisy', 'AE', 'VAE', 'U-Net']
fig, ax = plt.subplots(3, 5, figsize=(15, 9))
for r, k in enumerate(sel):
    imgs = [clean_eval[k,0], dirty_eval[k,0], pae[k,0], pvae[k,0], pun[k,0]]
    for cidx, im in enumerate(imgs):
        ax[r, cidx].imshow(im, cmap='inferno'); ax[r, cidx].axis('off')
        if r == 0: ax[r, cidx].set_title(cols[cidx])
plt.tight_layout(); plt.show()

## §5. Week 4 — Conditional DDPM diffusion (scaled, multi-GPU)

Scaled `DiffusionUNet` (ch=64, 4 levels, ~17.2M params) trained as a conditional DDPM:
predict the noise on the *clean* channel conditioned on the *dirty* channel, then denoise via
DDIM. `DenoisingDiffusion` auto-wraps in `nn.DataParallel`, so on a **T4 ×2** both GPUs are used.


In [ ]:
from src.data.dataset import create_dataloaders

# patch loaders in the [dirty, clean] 2-channel format the DDPM expects
d_tr, c_tr = dirty_all[train_idx], clean_all[train_idx]
d_va, c_va = dirty_all[val_idx],   clean_all[val_idx]
dif_train, dif_val = create_dataloaders(
    dirty_train=d_tr, clean_train=c_tr, dirty_val=d_va, clean_val=c_va,
    batch_size=16, num_workers=2, parse_patches=True, patch_size=PATCH_SIZE, n_patches=4)

cfg = default_diffusion_config(image_size=PATCH_SIZE)
# cfg.model.ch = 128   # scale up on the bigger Kaggle GPU if desired
diffusion = DenoisingDiffusion(cfg, device=str(device), lr=2e-5,
                               checkpoint_path='/kaggle/working/diffusion_best.pth.tar')
print('DDPM params:', sum(p.numel() for p in diffusion._core.parameters()),
      '| GPUs:', diffusion.num_gpus,
      '(DataParallel)' if diffusion.data_parallel else '(single)')

In [ ]:
res = diffusion.train(dif_train, dif_val, n_epochs=EPOCHS_DIFF)
plot_curves(res['train_losses'], res['val_losses'], 'Conditional DDPM (noise-estimation loss)')
diffusion.load_checkpoint('/kaggle/working/diffusion_best.pth.tar')

In [ ]:
# DDIM-denoise the shared eval patches, in batches
preds = []
for i in range(0, len(dirty_eval), 32):
    preds.append(diffusion.sample(dirty_eval[i:i+32], sampling_timesteps=DDIM_STEPS, use_ema=True).cpu())
ddpm_pred = torch.cat(preds)
RESULTS['DDPM (diffusion)'] = metrics(ddpm_pred, clean_eval)
print('DDPM eval:', RESULTS['DDPM (diffusion)'])

sel = np.random.default_rng(7).choice(len(dirty_eval), 4, replace=False)
fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for j, k in enumerate(sel):
    for r, (im, nm) in enumerate([(dirty_eval[k,0],'dirty'), (ddpm_pred[k,0],'DDPM'), (clean_eval[k,0],'clean')]):
        ax[r, j].imshow(im, cmap='inferno'); ax[r, j].axis('off')
        if j == 0: ax[r, j].set_title(nm, loc='left')
plt.tight_layout(); plt.show()

## §6. Unified comparison — all methods on the same eval patches (ranked by SSIM)


In [ ]:
import pandas as pd
df = (pd.DataFrame([(k, p, s, m) for k, (p, s, m) in RESULTS.items()],
                   columns=['Method', 'PSNR', 'SSIM', 'MSE'])
        .sort_values('SSIM', ascending=False).reset_index(drop=True))
df.to_csv('/kaggle/working/metrics_master.csv', index=False)
print(df.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].barh(df['Method'], df['SSIM']); ax[0].invert_yaxis(); ax[0].set_title('SSIM (higher better)')
ax[1].barh(df['Method'], df['PSNR']); ax[1].invert_yaxis(); ax[1].set_title('PSNR dB')
for a in ax: a.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

**Outputs** (notebook *Output* tab): `metrics_master.csv`, `diffusion_best.pth.tar`.

SSIM is the headline metric — it rewards preserving disk structure (rings/gaps) rather than the
blur that PSNR/MSE favour. Diffusion typically needs many epochs to overtake the regression models;
raise `EPOCHS_DIFF` (and optionally `cfg.model.ch`) for stronger samples.
